# EX_09 — Agentes con LangChain (ejercicios)

**Notebook de referencia:** `notebook/09_Agentes_LangChain.ipynb`

**Tiempo orientativo:** ~30 minutos.


## Actividad 1 — Tool decorator

Define un `@tool` (LangChain) que calcule el número de palabras de un texto. Prueba `.invoke` con un string.

*Hint:* `from langchain_core.tools import tool`.


In [6]:
from langchain_core.tools import tool

@tool
def count_words(text: str) -> int:
    """Count the number of words in a given text."""
    return len(text.split())

# TODO: invoke
count_words.invoke('Spiderman está en el tejado')


5

## Actividad 2 — Agente mínimo

Monta un agente (o `create_react_agent` según la versión de tu curso) con **un** LLM y la tool anterior. Si no tienes credenciales, deja el código comentado pero completo.


In [ ]:
from langchain.tools import tool
from langchain_groq import ChatGroq
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate

# 1. Creamos una herramienta súper sencilla
@tool
def calculadora_suma(a: int, b: int) -> int:
    """Suma dos números enteros. Úsala cuando el usuario te pida sumar."""
    return a + b

tools = [calculadora_suma]

# 2. Configuramos el LLM con Groq
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0, 
    api_key="Pon tu API Key aquí" 
)

# 3. El Prompt (Copiado exactamente con el estilo de tu imagen)
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente con acceso a herramientas. Usa las herramientas cuando sea necesario. Siempre responde en español."),
    ("placeholder", "{chat_history}"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

# 4. Construimos el Agente y el Ejecutor
agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("Agente creado ✓")

# --- PRUEBA DE EJECUCIÓN ---
# Como pusimos "chat_history" en el prompt, le pasamos una lista vacía por ahora.
resultado = agent_executor.invoke({
    "input": "Hola, ¿puedes sumarme 45 más 22?",
    "chat_history": [] 
    })
print("\nRespuesta:", resultado["output"])

Agente creado ✓


> Entering new AgentExecutor chain...

Invoking: `calculadora_suma` with `{'a': 45, 'b': 22}`


67El resultado de la suma es 67. ¿Necesitas algo más?

> Finished chain.

Respuesta: El resultado de la suma es 67. ¿Necesitas algo más?


## Actividad 3 — Traza deseada

Escribe la secuencia ideal de eventos (Thought / Action / Observation) para la pregunta: "How many words in this sentence: I love RAG?"


_Secuencia (markdown):_

1. **Pensamiento (Thought):** Necesito averiguar el número exacto de palabras en la frase "I love RAG?". Dado que los modelos de lenguaje a veces fallan al contar caracteres o palabras, usaré una herramienta para contar de forma exacta.
2. **Acción (Action):** `herramienta_contar_palabras` (con la entrada: "I love RAG?")
3. **Observación (Observation):** 3
4. **Pensamiento (Thought):** Ahora conozco la respuesta final basándome en el resultado de la herramienta. La frase tiene 3 palabras.
5. **Respuesta Final (Final Answer):** Hay 3 palabras en esta frase.